# Ddri Open-Meteo Weather Fetch

이 노트북은 강남구 기준 좌표에서 Open-Meteo 아카이브 API를 호출해 시간별 날씨 데이터를 수집하는 문서다.

## 이 노트북의 목적

- 2024-01-01 보완 데이터와 2025년 전체 날씨를 같은 방식으로 수집한다.
- 어떤 좌표와 어떤 변수 조합을 사용했는지 남긴다.
- 결과가 예측용 일 단위 날씨 테이블의 원천이 된다는 점을 명확히 한다.

## 1. 기본 설정

좌표는 강남구 기준점이며, API는 시간별 온도·습도·강수량·풍속을 요청한다. 이후 예측 파이프라인에서는 이 원본을 일 단위로 다시 집계해 사용한다.

In [1]:
from pathlib import Path

import pandas as pd
import requests

BASE_DIR = Path('/Users/cheng80/Desktop/ddri_work')
OUTPUT_DIR = BASE_DIR / 'works' / '02_data_collection' / '02_weather' / 'data'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LATITUDE = 37.514557
LONGITUDE = 127.0495556
BASE_URL = 'https://archive-api.open-meteo.com/v1/archive'

print('output dir:', OUTPUT_DIR)
print('latitude, longitude:', LATITUDE, LONGITUDE)

output dir: /Users/cheng80/Desktop/ddri_work/works/02_data_collection/02_weather/data
latitude, longitude: 37.514557 127.0495556


## 2. API 호출 함수 정의

이 함수는 시작일과 종료일을 받아 시간별 날씨를 DataFrame으로 반환한다. 컬럼은 예측 데이터셋 빌더가 그대로 읽을 수 있도록 통일한다.

In [2]:
def fetch_open_meteo_history(start_date, end_date):
    # Open-Meteo 아카이브 API에서 강남구 기준 좌표의 시간별 날씨를 요청한다.
    params = {
        'latitude': LATITUDE,
        'longitude': LONGITUDE,
        'start_date': start_date,
        'end_date': end_date,
        'hourly': 'temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m',
        'timezone': 'Asia/Seoul',
    }
    response = requests.get(BASE_URL, params=params, timeout=60)
    response.raise_for_status()
    data = response.json()
    hourly = data['hourly']
    df = pd.DataFrame(
        {
            'datetime': hourly['time'],
            'temperature': hourly['temperature_2m'],
            'humidity': hourly['relative_humidity_2m'],
            'precipitation': hourly['precipitation'],
            'wind_speed': hourly['wind_speed_10m'],
        }
    )
    df['datetime'] = pd.to_datetime(df['datetime'])
    return df

## 3. 저장 함수와 수집 범위

이번 프로젝트에서 실제로 필요한 범위는 두 가지였다.

- 2024-01-01 하루 보완
- 2025-01-01 ~ 2025-12-31 전체 수집

이 셀은 해당 범위를 파일로 저장한다.

In [3]:
def save_range(start_date, end_date, output_name):
    # 수집 결과를 프로젝트 표준 경로에 저장하고 행 수를 함께 출력한다.
    df = fetch_open_meteo_history(start_date, end_date)
    output_path = OUTPUT_DIR / output_name
    df.to_csv(output_path, index=False)
    print(f'saved: {output_path}')
    print(f'rows={len(df)} range={start_date}~{end_date}')

ranges = [
    ('2024-01-01', '2024-01-01', 'ddri_weather_2024_0101_hourly.csv'),
    ('2025-01-01', '2025-12-31', 'ddri_weather_2025_hourly.csv'),
]

for start_date, end_date, output_name in ranges:
    save_range(start_date, end_date, output_name)

saved: /Users/cheng80/Desktop/ddri_work/works/02_data_collection/02_weather/data/ddri_weather_2024_0101_hourly.csv
rows=24 range=2024-01-01~2024-01-01


saved: /Users/cheng80/Desktop/ddri_work/works/02_data_collection/02_weather/data/ddri_weather_2025_hourly.csv
rows=8760 range=2025-01-01~2025-12-31


## 4. 결과 확인

수집이 끝나면 파일이 실제로 생성되었는지, 행 수가 기대 범위와 맞는지 확인한다. 2025년은 윤년이 아니므로 시간별 기준 8760행이 기대치다.

In [4]:
for file_name in ['ddri_weather_2024_0101_hourly.csv', 'ddri_weather_2025_hourly.csv']:
    path = OUTPUT_DIR / file_name
    df = pd.read_csv(path)
    print(file_name, len(df))
    display(df.head(3))

ddri_weather_2024_0101_hourly.csv 24


,datetime,temperature,humidity,precipitation,wind_speed
0,2024-01-01 00:00:00,-1.0,94,0.0,4.3
1,2024-01-01 01:00:00,-2.8,95,0.0,7.9
2,2024-01-01 02:00:00,-3.3,95,0.0,9.3


ddri_weather_2025_hourly.csv 8760


,datetime,temperature,humidity,precipitation,wind_speed
0,2025-01-01 00:00:00,-3.3,71,0.0,1.5
1,2025-01-01 01:00:00,-3.4,72,0.0,1.3
2,2025-01-01 02:00:00,-3.4,72,0.0,2.6
